# TextGraphicalizer: first WordNet ontology pass

This notebook runs the Laya-backed `TextGraphicalizer` on a few made-up paragraphs using the checked-in high-level WordNet ontology. Each result is rendered inline as a directed NetworkX graph.

> The first model load downloads the pinned Laya checkpoint into the Hugging Face cache.

## One-time setup

This notebook is intended to run with the project’s Python 3.12 virtual environment. From the repository root, create it once and install the notebook extra.

In [1]:
# From the repository root, run once in a terminal:
# python3.12 -m venv .venv312
# .venv312/bin/python -m pip install -e ".[notebook]"
# .venv312/bin/python -m ipykernel install --user --name textgraphicalizer-py312 \
#     --display-name "TextGraphicalizer (Python 3.12)"

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display

assert sys.version_info[:2] == (3, 12), (
    f"Select the 'TextGraphicalizer (Python 3.12)' kernel; current interpreter is {sys.version}"
)
print(f"Using Python: {sys.executable}")

# Make the notebook work when launched from either the repository root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "ontology.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from textgraphicalizer import TextGraphicalizer

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 15,
    "font.size": 10,
})
ontology_path = ROOT / "ontology.yaml"
ontology_path

PosixPath('/Users/f.costa/Code/TextGraphicalizer/ontology.yaml')

In [3]:
extractor = TextGraphicalizer(
    ontology=ontology_path,
    node_threshold=0.45,
    edge_threshold=0.50,
    connected=False,
    max_node_degree=4,
    device="auto",
).fit()

ImportError: Laya is not installed. Install TextGraphicalizer with its project dependencies or run `pip install laya==0.1.6`.

## Made-up paragraphs

These examples intentionally mix people, organizations, locations, artifacts, substances, events, and causal language so the coarse ontology has several opportunities to fire.

In [ ]:
paragraphs = [
    (
        "Storm response",
        "After a heavy storm, the river flooded the village and forced the emergency organization to move residents to a nearby shelter.",
    ),
    (
        "Drought and harvest",
        "The research team published a report explaining that repeated drought reduced the harvest and changed the condition of the soil.",
    ),
    (
        "Robot workshop",
        "A university launched a program that taught students how to build a small robot from metal parts and use it in the laboratory.",
    ),
]

graphs = []
for title, paragraph in paragraphs:
    graph = extractor.transform(paragraph)
    graphs.append((title, paragraph, graph))
    print(f"{title}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
def render_graph(graph, title, paragraph):
    """Render one graph with probability-aware nodes and directed edges."""
    fig, ax = plt.subplots(figsize=(13, 6.5))
    fig.patch.set_facecolor("#f7f8fa")
    ax.set_facecolor("#f7f8fa")

    if graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "No concepts crossed the node threshold",
                ha="center", va="center", fontsize=13, color="#4b5563")
        ax.set_title(title)
        ax.axis("off")
        fig.text(0.02, 0.02, paragraph, ha="left", va="bottom",
                 fontsize=9, color="#4b5563", wrap=True)
        plt.show()
        return

    positions = nx.spring_layout(graph, seed=17, k=1.8 / max(graph.number_of_nodes() ** 0.5, 1))
    node_probabilities = [graph.nodes[node].get("probability", 0.0) for node in graph.nodes]
    node_sizes = [900 + 1100 * probability for probability in node_probabilities]
    edge_probabilities = [graph.edges[edge].get("probability", 0.0) for edge in graph.edges]
    edge_widths = [1.2 + 3.5 * probability for probability in edge_probabilities]

    nx.draw_networkx_nodes(
        graph, positions, ax=ax, node_size=node_sizes,
        node_color=node_probabilities, cmap=plt.cm.viridis,
        vmin=0.0, vmax=1.0, alpha=0.94,
        edgecolors="#263238", linewidths=1.3,
    )
    nx.draw_networkx_edges(
        graph, positions, ax=ax, arrows=True, arrowstyle="-|>",
        arrowsize=20, width=edge_widths or 1.5,
        edge_color=edge_probabilities or "#94a3b8",
        edge_cmap=plt.cm.plasma, edge_vmin=0.0, edge_vmax=1.0,
        connectionstyle="arc3,rad=0.08", min_source_margin=14, min_target_margin=18,
    )

    node_labels = {node: data.get("label", node) for node, data in graph.nodes(data=True)}
    nx.draw_networkx_labels(graph, positions, labels=node_labels, ax=ax,
                           font_size=9, font_weight="bold",
                           bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=1.5))
    edge_labels = {
        (source, target): f"{data.get('label', '')}  {data.get('probability', 0.0):.2f}"
        for source, target, data in graph.edges(data=True)
    }
    if edge_labels:
        nx.draw_networkx_edge_labels(
            graph, positions, edge_labels=edge_labels, ax=ax,
            font_size=8, rotate=False, label_pos=0.52,
            bbox=dict(facecolor="#fff7ed", edgecolor="#fed7aa", alpha=0.92, pad=2),
        )

    ax.set_title(f"{title}  ·  {graph.number_of_nodes()} nodes / {graph.number_of_edges()} edges", pad=16)
    ax.text(0.5, -0.08, paragraph, transform=ax.transAxes, ha="center", va="top",
            fontsize=9, color="#4b5563", wrap=True)
    ax.text(0.01, 0.01, "node color/size = node probability   ·   edge width/label = edge probability",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=8, color="#64748b")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

for title, paragraph, graph in graphs:
    render_graph(graph, title, paragraph)

## Inspect one graph as ordinary NetworkX data

The visualization is only a view over the normal NetworkX graph, so downstream algorithms can use the same nodes, edges, and evidence attributes.

In [ ]:
title, paragraph, graph = graphs[0]
print("Nodes:")
display(list(graph.nodes(data=True)))
print("Edges:")
display(list(graph.edges(data=True)))
print("Graph metadata:")
display(graph.graph)